In [1]:
# Install deps
!pip install pandas numpy

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import json
import os

In [3]:
raw_path = "/content/raw.csv"
df_raw = pd.read_csv(raw_path)

print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (12956, 9)


,Unnamed: 0.1,Unnamed: 0,title_ukr,url,source_text,source_lang,ukr_text,label,date
0,0,0.0,Десятки бригад! Операція Почалась – Новий наст...,https://puer-press.org.ua/admin/268/,Сили оборони України можуть перейти в новий на...,uk,Сили оборони України можуть перейти в новий на...,Fake,2022-12-13
1,1,1.0,Дуже важкий грип зараз мандрує Україною! У діт...,https://puer-press.org.ua/admin/265/,Подаємо мовою оригіналу: Тарас Жиравецький Дуж...,uk,Подаємо мовою оригіналу: Тарас Жиравецький Дуж...,Fake,2022-12-13
2,2,2.0,"Виcтyп гeнceкa НАТО nідірвaв мережу: “Цiнa, як...",https://puer-press.org.ua/admin/235/,"Шикapний виcтyп гeнceкa НАТО, в якoмy вiн зaкл...",uk,"Шикapний виcтyп гeнceкa НАТО, в якoмy вiн зaкл...",Fake,2022-07-19
3,3,3.0,"Залишилися лічені дні, почнеться справжня “м’я...",https://puer-press.org.ua/admin/227/,Вже восени для російських окупантів в Україні ...,uk,Вже восени для російських окупантів в Україні ...,Fake,2022-07-19
4,4,4.0,"Кремль втретє змінив тактику щодо України, теп...",https://puer-press.org.ua/anews/219/,Російські окупанти скоригували свої плани щодо...,uk,Російські окупанти скоригували свої плани щодо...,Fake,2022-07-06


In [4]:
import re
import unicodedata

UA_ABBREVIATIONS = [
    "м.", "вул.", "р.", "т.д.", "т.ін.", "ст.", "№"
]

def clean_text(text: str) -> str:
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'").replace("ʼ", "'")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def mask_pii(text: str) -> str:
    text = re.sub(r"http\S+|www\S+", "<URL>", text)
    text = re.sub(r"\S+@\S+", "<EMAIL>", text)
    text = re.sub(r"\+?\d[\d\s\-]{7,}\d", "<PHONE>", text)
    return text


def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def sentence_split(text: str) -> list:
    sentences = []
    buffer = ""

    tokens = text.split(" ")

    for token in tokens:
        buffer += token + " "

        if token.endswith("."):
            if any(token.endswith(abbrev) for abbrev in UA_ABBREVIATIONS):
                continue
            sentences.append(buffer.strip())
            buffer = ""

    if buffer.strip():
        sentences.append(buffer.strip())

    return sentences


def preprocess(text: str) -> dict:
    clean = clean_text(text)
    masked = mask_pii(clean)
    normalized = normalize_text(masked)
    sentences = sentence_split(normalized)

    return {
        "clean": normalized,
        "sentences": sentences
    }

In [5]:
df = df_raw.copy()

df["title_ukr"] = df["title_ukr"].fillna("")
df["ukr_text"] = df["ukr_text"].fillna("")

df["raw_text"] = df["title_ukr"] + " " + df["ukr_text"]

In [6]:
results = df["raw_text"].apply(preprocess)

df["processed_text"] = results.apply(lambda x: x["clean"])
df["sentences"] = results.apply(lambda x: x["sentences"])

In [7]:
df["raw_len"] = df["raw_text"].apply(len)
df["proc_len"] = df["processed_text"].apply(len)

print("Raw mean length:", df["raw_len"].mean())
print("Processed mean length:", df["proc_len"].mean())

Raw mean length: 855.6896418647731
Processed mean length: 855.0010033961099


In [8]:
dup_raw = df.duplicated(subset=["raw_text"]).sum()
dup_proc = df.duplicated(subset=["processed_text"]).sum()

print("Raw duplicates:", dup_raw)
print("Processed duplicates:", dup_proc)

Raw duplicates: 195
Processed duplicates: 196


In [9]:
short_raw = (df["raw_text"].str.split().str.len() < 5).sum()
short_proc = (df["processed_text"].str.split().str.len() < 5).sum()

print("Short raw:", short_raw)
print("Short processed:", short_proc)

Short raw: 10
Short processed: 10


In [10]:
df[["raw_text", "processed_text"]].sample(15)

,raw_text,processed_text
7914,Троє провідних депутатів Європейського парламе...,троє провідних депутатів європейського парламе...
3592,Зеленський звільнив Данілова з посади секрета...,зеленський звільнив данілова з посади секретар...
5163,Олекса Новаківський: геній світового рівня Пр...,олекса новаківський: геній світового рівня про...
9122,"Німеччина перетнула червону межу, коли почала ...","німеччина перетнула червону межу, коли почала ..."
4839,Еспресо спільно з платформою пам’яті Меморіал ...,еспресо спільно з платформою пам'яті меморіал ...
8089,"Так, добре. Так, добре. Тепер ще й пташиний грип.","так, добре. так, добре. тепер ще й пташиний грип."
6642,В Офісі президента заявили що Макрон приїде в...,в офісі президента заявили що макрон приїде в ...
11982,: Російські військові викрали медичне обладнан...,: російські військові викрали медичне обладнан...
4844,РФ увечері 19 квітня обстріляла селище Антонів...,рф увечері 19 квітня обстріляла селище антонів...
6318,Ексміністр оборони України Олексій Резніков с...,ексміністр оборони україни олексій резніков ст...


In [11]:
def test_idempotence(text):
    first = preprocess(text)["clean"]
    second = preprocess(first)["clean"]
    return first == second

tests = df["raw_text"].sample(50).apply(test_idempotence)
print("Idempotence passed:", tests.all())

Idempotence passed: True


In [12]:
empty_after = (df["processed_text"].str.strip() == "").sum()
print("Empty after processing:", empty_after)

Empty after processing: 3


In [13]:
os.makedirs("data/processed_v2", exist_ok=True)

df_out = df[["raw_text", "processed_text", "label"]]
df_out.to_csv("data/processed_v2/processed_v2.csv", index=False)

print("processed_v2.csv saved")

processed_v2.csv saved
